In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [2]:
with open(r"/home/user/Downloads/bahrainfinder_2026_07_04.json", "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

{"reference_number": "1252", "url": "https://bahrainfinder.bh/en/property/villa-for-sale-in-jid-al-haj-11585.html", "broker_display_name": "LAKUM Real Estates", "broker": "LAKUM REAL ESTATES", "category": "sale", "category_url": "https://bahrainfinder.bh/en/for-sale/", "title": "Residential Villa For Sale", "description": "Residential Villa for Sale in Jid Al HajArea: 254.5 sqmBuilt-up size: 230 sqmBathrooms: 2Halls: 1Floors: 2Furnishing: FurnishedCondition: NewParking spaces: 2Price: 129,000 BHDRoads: 1Features & Amenities: Garden, Closed Kitchen, Swimming PoolFor more information, do not hesitate to contact me now | Manaf Tareef – LAKUM Real Estates", "location": "Jid Al Haj", "price": "129,000", "currency": "BHD", "price_per": "", "bedrooms": "", "bathrooms": "2", "furnished": "Furnished", "rera_permit_number": "", "dtcm_licence": "", "scraped_ts": "2026-07-27", "amenities": "Garden, Closed Kitchen, Swimming Pool", "details": "254.5 m²230 m²", "agent_name": "Manaf Tareef", "number_o

In [4]:
import pandas as pd

# File configuration
file_path = "/home/user/Downloads/aqarmap_egp_2026_10_04.json"
chunk_size = 10_000

total_records = 0
same_values = 0
different_values = 0
missing_values = 0
broker_not_capital = 0

different_samples = []
capital_samples = []
max_samples = 10

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Processing chunk {chunk_no}: {chunk.shape}")
    total_records += len(chunk)

    # Check required columns
    required = ["broker", "broker_display_name"]
    if not all(col in chunk.columns for col in required):
        raise ValueError(f"Missing required columns: {required}")

    broker = chunk["broker"]
    display = chunk["broker_display_name"]

    # Check missing values
    missing_mask = broker.isna() | display.isna()
    missing_values += int(missing_mask.sum())

    # Compare values, ignoring leading/trailing spaces
    broker_clean = broker.astype("string").str.strip()
    display_clean = display.astype("string").str.strip()

    same_mask = (
        broker_clean.eq(display_clean).fillna(False)
        & ~missing_mask
    )
    different_mask = ~same_mask & ~missing_mask

    same_values += int(same_mask.sum())
    different_values += int(different_mask.sum())

    # Capture examples where values differ
    if len(different_samples) < max_samples:
        sample = chunk.loc[
            different_mask, ["broker", "broker_display_name"]
        ].head(max_samples - len(different_samples))

        different_samples.extend(sample.to_dict("records"))

    # Check whether broker is uppercase
    # Ignore nulls and non-string values
    string_mask = broker.map(lambda x: isinstance(x, str))
    uppercase_mask = broker.map(
        lambda x: isinstance(x, str) and x == x.upper()
    )

    not_capital_mask = string_mask & ~uppercase_mask
    broker_not_capital += int(not_capital_mask.sum())

    # Capture examples where broker is not uppercase
    if len(capital_samples) < max_samples:
        sample = chunk.loc[
            not_capital_mask, ["broker", "broker_display_name"]
        ].head(max_samples - len(capital_samples))

        capital_samples.extend(sample.to_dict("records"))

# Final report
print("\n" + "=" * 60)
print("BROKER FIELD QA REPORT")
print("=" * 60)

print(f"Total records: {total_records:,}")

print("\n1. broker vs broker_display_name")
print(f"Same values: {same_values:,}")
print(f"Different values: {different_values:,}")
print(f"Records with missing values: {missing_values:,}")

if different_samples:
    print("\nSample mismatched values:")
    print(pd.DataFrame(different_samples).to_string(index=False))

print("\n2. broker uppercase validation")
print(f"Non-uppercase string values: {broker_not_capital:,}")

if capital_samples:
    print("\nSample non-uppercase values:")
    print(pd.DataFrame(capital_samples).to_string(index=False))

print("\nFINAL STATUS")
if different_values == 0 and missing_values == 0 and broker_not_capital == 0:
    print("PASS: Both fields match and broker is uppercase.")
else:
    print("FAIL: Differences, missing values, or non-uppercase values found.")

Processing chunk 1: (10000, 37)
Processing chunk 2: (10000, 37)
Processing chunk 3: (10000, 37)
Processing chunk 4: (10000, 37)
Processing chunk 5: (10000, 37)
Processing chunk 6: (10000, 37)
Processing chunk 7: (10000, 37)
Processing chunk 8: (10000, 37)
Processing chunk 9: (10000, 37)
Processing chunk 10: (10000, 37)
Processing chunk 11: (10000, 37)
Processing chunk 12: (10000, 37)
Processing chunk 13: (10000, 37)
Processing chunk 14: (10000, 37)
Processing chunk 15: (10000, 37)
Processing chunk 16: (10000, 37)
Processing chunk 17: (10000, 37)
Processing chunk 18: (10000, 37)
Processing chunk 19: (10000, 37)
Processing chunk 20: (10000, 37)
Processing chunk 21: (10000, 37)
Processing chunk 22: (10000, 37)
Processing chunk 23: (10000, 37)
Processing chunk 24: (10000, 37)
Processing chunk 25: (10000, 37)
Processing chunk 26: (10000, 37)
Processing chunk 27: (10000, 37)
Processing chunk 28: (10000, 37)
Processing chunk 29: (6413, 37)

BROKER FIELD QA REPORT
Total records: 286,413

1. br

In [2]:
import pandas as pd

file_path = "/home/user/Downloads/dubizzle_uae_2026_10_04.json"
chunk_size = 10_000

issue_examples = {}

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Processing chunk {chunk_no}")

    for col in chunk.columns:

        # Keep only the first issue per field
        if col in issue_examples:
            continue

        string_mask = chunk[col].map(lambda x: isinstance(x, str))
        values = chunk.loc[string_mask, col].astype("string")

        if values.empty:
            continue

        checks = [
            ("Leading whitespace", values.str.match(r"^\s+")),
            ("Trailing whitespace", values.str.match(r"\s+$")),
            (
                "Multiple consecutive spaces",
                values.str.contains(r" {2,}", regex=True, na=False)
            ),
            (
                "Tabs or newlines",
                values.str.contains(r"[\t\r\n]", regex=True, na=False)
            )
        ]

        for issue_type, mask in checks:
            if mask.any():
                idx = values.index[mask][0]

                record_no = (
                    (chunk_no - 1) * chunk_size
                    + chunk.index.get_loc(idx) + 1
                )

                issue_examples[col] = {
                    "record_no": record_no,
                    "url": chunk.loc[idx, "url"]
                          if "url" in chunk.columns else None,
                    "issue": issue_type
                }
                break

print("\n========== WHITESPACE QA REPORT ==========")

if issue_examples:
    report = pd.DataFrame([
        {"field": field, **details}
        for field, details in issue_examples.items()
    ])

    print(report.to_string(index=False))
    print(f"\nTotal fields with whitespace issues: {len(report)}")
else:
    print("No whitespace issues found.")

Processing chunk 1
Processing chunk 2
Processing chunk 3
Processing chunk 4
Processing chunk 5
Processing chunk 6
Processing chunk 7
Processing chunk 8
Processing chunk 9
Processing chunk 10
Processing chunk 11
Processing chunk 12
Processing chunk 13
Processing chunk 14
Processing chunk 15
Processing chunk 16
Processing chunk 17
Processing chunk 18
Processing chunk 19
Processing chunk 20
Processing chunk 21
Processing chunk 22
Processing chunk 23
Processing chunk 24
Processing chunk 25
Processing chunk 26
Processing chunk 27
Processing chunk 28
Processing chunk 29
Processing chunk 30
Processing chunk 31
Processing chunk 32
Processing chunk 33
Processing chunk 34
Processing chunk 35
Processing chunk 36
Processing chunk 37
Processing chunk 38
Processing chunk 39
Processing chunk 40
Processing chunk 41
Processing chunk 42
Processing chunk 43
Processing chunk 44
Processing chunk 45
Processing chunk 46
Processing chunk 47
Processing chunk 48
Processing chunk 49
Processing chunk 50
Processin

In [1]:
import pandas as pd
import json

file_path = "/home/user/Downloads/dubizzle_uae_2026_10_04.json"
chunk_size = 10_000

for chunk_no, chunk in enumerate(
    pd.read_json(file_path, lines=True, chunksize=chunk_size),
    start=1
):
    print(f"Chunk {chunk_no}: {chunk.shape}")

    # Convert list/dict columns only when needed for QA
    for col in chunk.columns:
        if chunk[col].map(lambda x: isinstance(x, (list, dict))).any():
            chunk[col] = chunk[col].map(
                lambda x: json.dumps(x, sort_keys=True, ensure_ascii=False)
                if isinstance(x, (list, dict))
                else x
            )

    # Perform your QA checks on this chunk here

Chunk 1: (10000, 42)
Chunk 2: (10000, 42)
Chunk 3: (10000, 42)
Chunk 4: (10000, 42)
Chunk 5: (10000, 42)
Chunk 6: (10000, 42)
Chunk 7: (10000, 42)
Chunk 8: (10000, 42)
Chunk 9: (10000, 42)
Chunk 10: (10000, 42)
Chunk 11: (10000, 42)
Chunk 12: (10000, 42)
Chunk 13: (10000, 42)
Chunk 14: (10000, 42)
Chunk 15: (10000, 42)
Chunk 16: (10000, 42)
Chunk 17: (10000, 42)
Chunk 18: (10000, 42)
Chunk 19: (10000, 42)
Chunk 20: (10000, 42)
Chunk 21: (10000, 42)
Chunk 22: (10000, 42)
Chunk 23: (10000, 42)
Chunk 24: (10000, 42)
Chunk 25: (10000, 42)
Chunk 26: (10000, 42)
Chunk 27: (10000, 42)
Chunk 28: (10000, 42)
Chunk 29: (10000, 42)
Chunk 30: (10000, 42)
Chunk 31: (10000, 42)
Chunk 32: (10000, 42)
Chunk 33: (10000, 42)
Chunk 34: (10000, 42)
Chunk 35: (10000, 42)
Chunk 36: (10000, 42)
Chunk 37: (10000, 42)
Chunk 38: (10000, 42)
Chunk 39: (10000, 42)
Chunk 40: (10000, 42)
Chunk 41: (10000, 42)
Chunk 42: (10000, 42)
Chunk 43: (10000, 42)
Chunk 44: (10000, 42)
Chunk 45: (10000, 42)
Chunk 46: (10000, 4

In [3]:
import pandas as pd
import json
import os
import re
import random
from datetime import datetime

# ============================================================
# 1. CONFIGURATION
# ============================================================

file_path = "/home/user/Downloads/dubizzle_uae_2026_10_04.json"

expected_fields = """
url
reference_number
id
broker_display_name
broker
category
category_url
title
property_type
sub_category_1
sub_category_2
depth
description
location
price
currency
price_per
bedrooms
bathrooms
furnished
rera_permit_number
dtcm_licence
scraped_ts
amenities
details
agent_name
number_of_photos
user_id
phone_number
date
iteration_number
published_at
completion_status
ded_license_number
rera_registration_number
latitude
longitude
package_type
last_update
listed_by
verified
trakheesi_url
""".strip().splitlines()

expected_empty_fields = [
    "dtcm_licence",
    "user_id",
    "listed_by",
    "ded_license_number"
]

allowed_values = {
    "price_per": {
        "", "Bi-Yearly", "Monthly", "Quarterly",
        "Yearly", "Daily", "سنويا"
    },
    "currency": {"AED"},
    "furnished": {"", "Yes"},
    "category": {"rent", "sale"},
    "package_type": {"standard", "featured", "premium"},
    "sub_category_1": {"commercial", "residential"},
    "completion_status": {"", "Ready", "Off-plan"},
    "verified": {"", "Checked", "TruCheckTM"}
}

date_fields = [
    "scraped_ts", "date", "published_at", "last_update"
]

# Update this to the expected run ID for the specific dataset.
expected_run_id = "propertyfinder-weekly:dubizzle_uae:2026junw3:qa"

# The provided iteration_number example is 2026_06_03.
iteration_pattern = re.compile(r"^\d{4}_\d{2}_\d{2}$")

chunk_size = 10_000
random_check_count = 10

output_path = file_path.replace(".json", "_qa_report.txt")

# ============================================================
# 2. HELPERS
# ============================================================

def is_empty(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    return pd.isna(value)


def is_valid_date(value):
    if is_empty(value):
        return True  # Missing values handled separately if required.

    try:
        datetime.strptime(str(value).strip(), "%Y-%m-%d")
        return True
    except ValueError:
        return False


def add_issue(issues, row_no, field, value, message):
    issues.append({
        "row_no": row_no,
        "field": field,
        "value": str(value)[:300],
        "issue": message
    })


# ============================================================
# 3. FILE NAME AND FORMAT CHECKS
# ============================================================

issues = []
summary = []

file_name = os.path.basename(file_path)

summary.append("DUBIZZLE UAE QA REPORT")
summary.append("=" * 60)
summary.append(f"File name: {file_name}")

if not file_name.endswith(".json"):
    summary.append("FAIL: File format should be JSON.")
else:
    summary.append("PASS: File extension is JSON.")

# Accepts names such as dubizzle_uae_2026_10_02.json
file_name_pattern = r"^dubizzle_uae_\d{4}_\d{2}_\d{2}\.json$"

if re.match(file_name_pattern, file_name):
    summary.append("PASS: File name follows the accepted naming pattern.")
else:
    summary.append(
        "FAIL: File name does not follow dubizzle_uae_YYYY_MM_DD.json."
    )

if "_uae_" in file_name:
    summary.append("PASS: Country suffix '_uae_' is present.")
else:
    summary.append("FAIL: Country suffix '_uae_' is missing.")

# ============================================================
# 4. CHUNK-BASED DATA QA
# ============================================================

total_rows = 0
seen_ids = set()
random_records = []
columns_found = None
chunk_count = 0

# Reservoir sampling selects 10 random records across the whole file.
rng = random.Random(42)

try:
    reader = pd.read_json(
        file_path,
        lines=True,
        chunksize=chunk_size
    )

    for chunk_no, chunk in enumerate(reader, start=1):
        chunk_count += 1
        chunk_start = total_rows
        chunk = chunk.reset_index(drop=True)

        if columns_found is None:
            columns_found = list(chunk.columns)

        total_rows += len(chunk)

        print(f"Processing chunk {chunk_no}: {len(chunk):,} rows")

        # ----------------------------------------------------
        # A. RANDOM SAMPLE OF 10 RECORDS
        # ----------------------------------------------------

        for idx, record in chunk.iterrows():
            record_no = chunk_start + idx + 2
            item = {
                "row_no": record_no,
                "record": record.to_dict()
            }

            if len(random_records) < random_check_count:
                random_records.append(item)
            else:
                position = rng.randint(0, record_no - 2)
                if position < random_check_count:
                    random_records[position] = item

        # ----------------------------------------------------
        # B. EMPTY EXPECTED FIELDS
        # ----------------------------------------------------

        for field in expected_empty_fields:
            if field not in chunk.columns:
                continue

            mask = ~chunk[field].map(is_empty)

            for idx in chunk.index[mask]:
                add_issue(
                    issues,
                    chunk_start + idx + 2,
                    field,
                    chunk.at[idx, field],
                    "Expected empty field but value is populated"
                )

        # ----------------------------------------------------
        # C. ALLOWED VALUES
        # ----------------------------------------------------

        for field, allowed in allowed_values.items():
            if field not in chunk.columns:
                continue

            for idx, value in chunk[field].items():
                if is_empty(value):
                    normalized = ""
                else:
                    normalized = str(value).strip()

                if normalized not in allowed:
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        field,
                        value,
                        f"Unexpected value; allowed values: {sorted(allowed)}"
                    )

        # ----------------------------------------------------
        # D. BROKER AND BROKER_DISPLAY_NAME COMPARISON
        # ----------------------------------------------------

        if {"broker", "broker_display_name"}.issubset(chunk.columns):
            for idx, row in chunk.iterrows():
                broker = row["broker"]
                display = row["broker_display_name"]

                if not is_empty(broker) and not is_empty(display):
                    if str(broker).strip() != str(display).strip():
                        add_issue(
                            issues,
                            chunk_start + idx + 2,
                            "broker",
                            broker,
                            "broker and broker_display_name do not match"
                        )

                if isinstance(broker, str) and broker != broker.upper():
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        "broker",
                        broker,
                        "broker value is not in uppercase"
                    )

        # ----------------------------------------------------
        # E. DATE FORMATS
        # ----------------------------------------------------

        for field in date_fields:
            if field not in chunk.columns:
                continue

            for idx, value in chunk[field].items():
                if not is_valid_date(value):
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        field,
                        value,
                        "Invalid date format; expected YYYY-MM-DD"
                    )

        # ----------------------------------------------------
        # F. ITERATION NUMBER
        # ----------------------------------------------------

        if "iteration_number" in chunk.columns:
            for idx, value in chunk["iteration_number"].items():
                if not is_empty(value) and not iteration_pattern.match(
                    str(value).strip()
                ):
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        "iteration_number",
                        value,
                        "Invalid format; expected YYYY_MM_week, e.g. 2026_06_03"
                    )

        # ----------------------------------------------------
        # G. DUPLICATE DATASET CHECK
        # ----------------------------------------------------

        if "id" in chunk.columns:
            for idx, value in chunk["id"].items():
                if is_empty(value):
                    continue

                key = str(value).strip()

                if key in seen_ids:
                    add_issue(
                        issues,
                        chunk_start + idx + 2,
                        "id",
                        value,
                        "Duplicate ID"
                    )
                else:
                    seen_ids.add(key)

        # ----------------------------------------------------
        # H. CURRENCY CHECK: AED
        # ----------------------------------------------------

        # Already covered by allowed_values.
        # Add more requirements here if needed.

# ============================================================
# 5. FINAL VALIDATION
# ============================================================

    summary.append("")
    summary.append(f"Total records: {total_rows:,}")
    summary.append(f"Total chunks processed: {chunk_count}")

    if 170_000 <= total_rows <= 190_000:
        summary.append("PASS: Record count is within the standard range.")
    elif total_rows >= 495_900:
        summary.append("INFO: Record count is within the latest dataset scale.")
    else:
        summary.append("CHECK: Record count is outside the specified ranges.")

    # --------------------------------------------------------
    # REQUIRED FIELDS AND FIELD NAME CASE
    # --------------------------------------------------------

    if columns_found is not None:
        missing_fields = sorted(set(expected_fields) - set(columns_found))
        unexpected_fields = sorted(set(columns_found) - set(expected_fields))

        if not missing_fields:
            summary.append("PASS: All expected fields are present.")
        else:
            summary.append(f"FAIL: Missing fields: {missing_fields}")

        if unexpected_fields:
            summary.append(f"CHECK: Unexpected fields: {unexpected_fields}")

        wrong_case = [
            col for col in columns_found
            if col != col.lower()
        ]

        if wrong_case:
            summary.append(f"FAIL: Field names not lowercase: {wrong_case}")
        else:
            summary.append("PASS: All field names are lowercase.")

        duplicate_column_names = (
            pd.Index(columns_found)[pd.Index(columns_found).duplicated()]
            .tolist()
        )

        if duplicate_column_names:
            summary.append(
                f"FAIL: Duplicate column names: {duplicate_column_names}"
            )

    # --------------------------------------------------------
    # UNIQUE DATASET CHECK
    # --------------------------------------------------------

    if "id" in (columns_found or []):
        if total_rows == len(seen_ids):
            summary.append("PASS: Dataset IDs are unique.")
        else:
            summary.append(
                f"FAIL: Duplicate or missing IDs detected. "
                f"Rows: {total_rows:,}; unique IDs: {len(seen_ids):,}"
            )

    # --------------------------------------------------------
    # RANDOM RECORD OUTPUT
    # --------------------------------------------------------

    summary.append("")
    summary.append("RANDOM DATA CHECK (10 RECORDS)")
    summary.append("-" * 60)

    for sample in sorted(random_records, key=lambda x: x["row_no"]):
        summary.append(f"Row {sample['row_no']}:")
        summary.append(
            json.dumps(
                sample["record"],
                ensure_ascii=False,
                default=str
            )
        )

    # --------------------------------------------------------
    # ISSUE REPORT
    # --------------------------------------------------------

    summary.append("")
    summary.append(f"Total QA issues found: {len(issues):,}")
    summary.append("")

    if issues:
        issue_df = pd.DataFrame(issues)
        issue_df.to_csv(
            file_path.replace(".json", "_qa_issues.csv"),
            index=False
        )
        summary.append(
            "Detailed issues saved to: "
            + file_path.replace(".json", "_qa_issues.csv")
        )
    else:
        summary.append("No issues found by the configured checks.")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(summary))

    print("\n".join(summary[:15]))
    print(f"\nQA summary saved to: {output_path}")

except Exception as e:
    print(f"QA failed: {type(e).__name__}: {e}")

Processing chunk 1: 10,000 rows
Processing chunk 2: 10,000 rows
Processing chunk 3: 10,000 rows
Processing chunk 4: 10,000 rows
Processing chunk 5: 10,000 rows
Processing chunk 6: 10,000 rows
Processing chunk 7: 10,000 rows
Processing chunk 8: 10,000 rows
Processing chunk 9: 10,000 rows
Processing chunk 10: 10,000 rows
Processing chunk 11: 10,000 rows
Processing chunk 12: 10,000 rows
Processing chunk 13: 10,000 rows
Processing chunk 14: 10,000 rows
Processing chunk 15: 10,000 rows
Processing chunk 16: 10,000 rows
Processing chunk 17: 10,000 rows
Processing chunk 18: 10,000 rows
Processing chunk 19: 10,000 rows
Processing chunk 20: 10,000 rows
Processing chunk 21: 10,000 rows
Processing chunk 22: 10,000 rows
Processing chunk 23: 10,000 rows
Processing chunk 24: 10,000 rows
Processing chunk 25: 10,000 rows
Processing chunk 26: 10,000 rows
Processing chunk 27: 10,000 rows
Processing chunk 28: 10,000 rows
Processing chunk 29: 10,000 rows
Processing chunk 30: 10,000 rows
Processing chunk 31

In [4]:
#===================================================================================

In [60]:
import pandas as pd
import json

file_path = "/home/user/Downloads/bayut_uae_2026_11_02.json"

try:
    df = pd.read_json(file_path, lines=True)
except ValueError:
    try:
        df = pd.read_json(file_path, lines=False)
    except ValueError:
        with open(file_path, "r", encoding="utf-8-sig") as f:
            data = json.load(f)

        df = pd.DataFrame(data)



: 

In [32]:
df.shape

(1138, 19)

In [36]:
df.date.value_counts()

date
2026-10-31    1138
Name: count, dtype: int64

In [34]:
df.furnished.value_counts()

AttributeError: 'DataFrame' object has no attribute 'furnished'

In [9]:
df.published_at.value_counts()

published_at
2026-10-26    49
2026-10-21    33
2026-10-25    31
2026-10-30    30
2026-10-14    30
2026-01-11    30
2026-10-31    29
2026-10-23    28
2026-10-15    28
2026-10-27    27
2026-10-24    27
2026-10-17    27
2026-10-19    26
2026-10-20    26
2026-10-22    24
2026-12-10    23
2026-10-18    22
2026-10-13    22
2026-03-10    22
2026-06-10    20
2026-08-10    20
2026-10-10    19
2026-04-10    19
2026-07-10    19
2026-10-16    18
2026-05-10    18
2026-10-29    18
2026-10-28    17
2026-09-10    17
2026-11-10    17
2026-09-15     8
2026-08-09     8
2026-02-10     8
2026-09-13     7
2026-11-09     7
2026-09-18     6
2026-09-16     6
2026-09-24     6
2026-12-09     5
2026-09-19     5
2026-09-26     5
2026-01-10     5
2026-09-28     4
2026-09-27     4
2026-09-23     4
2026-09-25     4
2026-04-09     3
2026-09-29     3
2026-09-21     3
2026-05-09     3
2026-09-14     3
2026-09-09     3
2026-09-17     3
2026-09-22     3
2026-07-09     2
2026-09-30     2
2026-06-09     2
2026-02-11     1
2

In [6]:
df.category.value_counts()

AttributeError: 'DataFrame' object has no attribute 'category'

In [15]:
import pandas as pd

df["date_transaction_nk"] = pd.to_datetime(
    df["date_transaction_nk"], errors="coerce"
)

start_date = "2026-10-14"
end_date = "2026-10-21"

# Get records outside the expected date range
issues = df[
    ~df["date_transaction_nk"].between(start_date, end_date)
    & df["date_transaction_nk"].notna()
]

# Show unique date and hash ID combinations
result = (
    issues[["date_transaction_nk", "transaction_hash_id"]]
    .drop_duplicates()
    .sort_values("date_transaction_nk", ascending=False)
)

print("Unique dates outside the expected range:")
print(result.to_string(index=False))

# Overall date range in the dataset
print("\nOverall date range in dataset:")
print("Start date:", df["date_transaction_nk"].min())
print("End date:", df["date_transaction_nk"].max())

# Date range of the issues
if not issues.empty:
    print("\nDate range of issues:")
    print("Start date:", issues["date_transaction_nk"].min())
    print("End date:", issues["date_transaction_nk"].max())

Unique dates outside the expected range:
date_transaction_nk transaction_hash_id
         2026-10-28    1337957b1257a00e
         2026-10-28    29a2eadc03115492
         2026-10-28    252202487d5ace18
         2026-10-28    433ffdf7a8c06001
         2026-10-27     5341739e890157c
         2026-10-27    250dd35ba57a8d51
         2026-10-27    3ab2c12a24eea0cc
         2026-10-27    11c210e9492634f6
         2026-10-27     949ef7343e13e68
         2026-10-27    3d96895cd50ec9a5
         2026-10-27    735921e7a1b7d0b2
         2026-10-27     abf6c30f63bb8d5
         2026-10-27    3e00c70fdc9605ce
         2026-10-27    436850b7ab43129d
         2026-10-27    796547d3bdc4fc82
         2026-10-27    42710643d8dd5d36
         2026-10-27    7a98902702edc423
         2026-10-27    18afd052de8b32d5
         2026-10-27     4f3811bff0e3d2a
         2026-10-27    1d8652ead6a8bb86
         2026-10-27    28fcb4b22ac37c78
         2026-10-27    4bb73bcaf50f9310
         2026-10-27    187513debb47c7bd

In [37]:
import pandas as pd

# Count actual null/NaN values
null_count = df.isna().sum()

# Count empty strings and strings containing only whitespace
empty_count = df.apply(
    lambda col: col.map(
        lambda x: isinstance(x, str) and x.strip() == ""
    ).sum()
)

# Combine counts
result = pd.DataFrame({
    "null_count": null_count,
    "empty_string_count": empty_count,
    "total_empty_or_null": null_count + empty_count
})


# Show fields containing empty or null values
result = result[result["total_empty_or_null"] > 0]
result = result.sort_values("total_empty_or_null", ascending=False)

print(result.to_string())

                                         null_count  empty_string_count  total_empty_or_null
broker_type                                       0                1138                 1138
mobile_number                                     0                1138                 1138
email_address                                     0                1138                 1138
the_unified_number_of_the_establishment           0                1138                 1138
street                                            0                1138                 1138
city                                              0                1138                 1138
district                                          0                1138                 1138
additional_number                                 0                 627                  627
building_number                                   0                  89                   89
region                                            0                   

In [17]:
# Find duplicate transaction_hash_id values
duplicates = df[df.duplicated(subset=["transaction_hash_id"], keep=False)]

print("Total duplicate records:", len(duplicates))
print("Duplicate transaction_hash_id count:",
      duplicates["transaction_hash_id"].nunique())

display(
    duplicates[["transaction_hash_id", "date_transaction_nk"]]
    .sort_values("transaction_hash_id")
)

Total duplicate records: 0
Duplicate transaction_hash_id count: 0


,transaction_hash_id,date_transaction_nk


In [34]:
#=========================================================================================================

In [27]:
import pandas as pd
import json

df = pd.read_json(
    r"/home/user/Downloads/ ajmanded_uae_2026_11.json",
    lines=True
)

# Convert list/dict columns into strings
df = df.apply(
    lambda col: col.map(
        lambda x: json.dumps(x, sort_keys=True)
        if isinstance(x, (list, dict))
        else x
    )
)

FileNotFoundError: File /home/user/Downloads/ ajmanded_uae_2026_11.json does not exist

In [22]:
import pandas as pd
import json

file_path = "/home/user/Downloads/bhgre_2026_11_05.json"

# Load JSON array
df = pd.read_json(file_path, orient="records")

# Convert list/dict columns into strings
for col in df.columns:
    if df[col].map(lambda x: isinstance(x, (list, dict))).any():
        df[col] = df[col].map(
            lambda x: json.dumps(x, sort_keys=True)
            if isinstance(x, (list, dict))
            else x
        )

print("Total records:", len(df))
print("Total columns:", len(df.columns))

ValueError: Trailing data

In [2]:
df.shape

(72883, 38)

In [13]:
import pandas as pd

# Count null + empty string values in each column
empty_counts = {}

for col in df.columns:
    # Null values
    null_count = df[col].isna().sum()

    # Empty strings / whitespace-only strings
    empty_string_count = (
        df[col]
        .fillna('')
        .astype(str)
        .str.strip()
        .eq('')
        .sum()
    )

    # Total missing/empty
    total = null_count + empty_string_count

    empty_counts[col] = {
        'null_count': null_count,
        'empty_string_count': empty_string_count,
        'total_null_or_empty': total
    }

result = pd.DataFrame(empty_counts).T

print(result.to_string())

                     null_count  empty_string_count  total_null_or_empty
reference_number              0                 735                  735
id                            0                   0                    0
url                           0                   0                    0
broker_display_name           0                   0                    0
broker                        0                   0                    0
category                      0                   0                    0
category_url                  0                   0                    0
title                         0                   0                    0
description                   0                   0                    0
location                      0                   0                    0
price                         0                   4                    4
currency                      0                   4                    4
price_per                     0                 349

In [38]:
df.bathrooms.value_counts()

AttributeError: 'DataFrame' object has no attribute 'bathrooms'

In [7]:
df[df.bathrooms == "+9"].url

15011            https://wasalt.com/en/property/sale/villa-544-sqm-with-3-apartments-facing-west-5895760
15549    https://wasalt.com/en/property/sale/villa-721-sqm-facing-north-west-on-20m-width-street-5875733
27472            https://wasalt.com/en/property/sale/villa-480-sqm-with-2-apartments-facing-east-5861344
Name: url, dtype: object

In [12]:
df.shape

(3048, 33)

In [39]:
import json

duplicate_count = (
    df.applymap(
        lambda x: json.dumps(x, sort_keys=True)
        if isinstance(x, (list, dict))
        else x
    )
    .duplicated()
    .sum()
)

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


/tmp/ipykernel_14475/600954025.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df.applymap(


In [7]:
df.duplicated().sum()

np.int64(0)

In [62]:
empty_cols = df.columns[df.isna().all()].tolist()
empty_cols

[]

In [63]:
df.columns


Index(['profile_url', 'first_name', 'middle_name', 'last_name', 'image_url',
       'office_name', 'address', 'description', 'languages', 'social',
       'website', 'email', 'title', 'country', 'city', 'zipcode', 'state',
       'agent_phone_numbers', 'office_phone_numbers'],
      dtype='object')

In [40]:
req_cols = """unique_id, url, name, license_number, license_expiry_date, broker_type, mobile_number, email_address, the_unified_number_of_the_establishment, broker_licenses, street, city, region, district, building_number, additional_number, scraped_ts, date, 
iteration_number"""
req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

missing = [c for c in req_cols if c not in df.columns]
extra = [c for c in df.columns if c not in req_cols]

print("Requirement columns:", len(req_cols))
print("File columns:", len(df.columns))
print("Missing:", missing)
print("Extra:", extra)
print("Order matches:", req_cols == list(df.columns))

Requirement columns: 19
File columns: 19
Missing: []
Extra: []
Order matches: True


In [58]:
print(df.columns.tolist())

['reference_number', 'id', 'url', 'broker_display_name', 'broker', 'category', 'category_url', 'title', 'description', 'location', 'price', 'currency', 'price_per', 'bedrooms', 'bathrooms', 'furnished', 'rera_permit_number', 'dtcm_licence', 'scraped_ts', 'amenities', 'details', 'agent_name', 'number_of_photos', 'user_id', 'phone_number', 'date', 'iteration_number', 'rera_registration_number', 'ded_license_number', 'rera_length', 'latitude', 'longitude', 'listing_id', 'package_type', 'locality', 'object_id', 'completion_status', 'ad_type', 'verified', 'sub_category_1', 'sub_category_2', 'property_type', 'depth', 'published_at', 'listing_availability', 'brn_registration_number', 'orn_number', 'bln_number', 'adm_number', '3d_live', '3d_image', '2d_image', 'ded_number_2', 'ded_number_3', 'ded_number_4', 'trucheck_on', 'email', 'trubroker', 'rating', 'no_of_reviews', 'property_of_the_week']


In [41]:
broker = df["broker"].fillna("").astype(str).str.strip()
display_name = df["broker_display_name"].fillna("").astype(str).str.strip()

# Find invalid rows
invalid = df[
    broker != display_name.str.upper()
][["id", "url", "broker", "broker_display_name"]]

print(f"Invalid rows: {len(invalid)}")
display(invalid)

KeyError: 'broker'

In [42]:
df.category.value_counts()

AttributeError: 'DataFrame' object has no attribute 'category'

In [68]:
df[df.bedrooms == "03"].url

AttributeError: 'DataFrame' object has no attribute 'bedrooms'

In [69]:
duplicates = df[df.duplicated(subset="url", keep=False)]

print(f"Duplicate URLs: {len(duplicates)}")
display(duplicates.sort_values("url"))

KeyError: Index(['url'], dtype='object')

In [70]:
df1 = df.copy()

for col in df1.columns:
    if df1[col].apply(lambda x: isinstance(x, list)).any():
        df1[col] = df1[col].astype(str)

duplicates = df1[df1.duplicated(keep=False)]

print(f"Duplicate rows: {len(duplicates)}")
display(duplicates)

TypeError: unhashable type: 'dict'

In [43]:
import pandas as pd

df1 = df.replace(r'^\s*$', pd.NA, regex=True)

empty_columns = df1.columns[df1.isna().all()]

print("Completely empty columns:")
print(list(empty_columns))

Completely empty columns:
['broker_type', 'mobile_number', 'email_address', 'the_unified_number_of_the_establishment', 'street', 'city', 'district']


In [44]:
import re

url_pattern = re.compile(
    r'^(https?://)'
    
    r'([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}'
    r'(/[^\s]*)?$'
)

invalid_url = df[
    ~df["url"].fillna("").astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_url)}")
display(invalid_url[[ "url"]])

Invalid URLs: 0


,url


In [29]:
df.depth.value_counts()

AttributeError: 'DataFrame' object has no attribute 'depth'

In [30]:
df.dtypes

profile_url             object
first_name              object
middle_name             object
last_name               object
image_url               object
office_name             object
address                 object
description             object
languages               object
social                  object
website                 object
email                   object
title                   object
country                 object
city                    object
zipcode                 object
state                   object
agent_phone_numbers     object
office_phone_numbers    object
dtype: object

In [25]:
phone_pattern = re.compile(r'^\+?\d+$')

invalid_phone = df[
    ~df["phone_number"].fillna("").astype(str).str.match(phone_pattern)
]

display(invalid_phone[["id", "phone_number"]])

,id,phone_number
0,289323058,373353XX
1,289477704,388723XX
2,289391100,333488XX
3,289386638,336942XX
4,289344122,393030XX
...,...,...
856,288067610,332202XX
857,288061648,396879XX
858,288057804,331744XX
859,287645032,340595XX


In [32]:
df.sub_category_1.value_counts()

AttributeError: 'DataFrame' object has no attribute 'sub_category_1'

In [74]:
import pandas as pd

details = df["details"].fillna("").astype(str).str.strip()

# Must end with: number + space + M²
mask = details.ne("") & ~details.str.fullmatch(r"\d+(?:\.\d+)? M²", na=False)

result = df.loc[mask, ["url", "details"]].copy()
result["record_no"] = result.index + 2

print("Invalid details:", len(result))
print(result[["record_no", "url", "details"]].to_string(index=False))

KeyError: 'details'

In [14]:
import re

email_pattern = re.compile(
    r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
)

invalid_emails = df[
    df["email"].notna() &
    (df["email"].astype(str).str.strip() != "") &
    ~df["email"].astype(str).str.strip().str.match(email_pattern)
]

print(f"Invalid emails: {len(invalid_emails)}")
display(invalid_emails[["email"]])

KeyError: 'email'

In [34]:
duplicates = df[df.duplicated("id", keep=False)]

print(f"Duplicate IDs: {duplicates.shape[0]}")
display(duplicates[["id"]])

KeyError: Index(['id'], dtype='object')

In [46]:
duplicates = (
    df[df["url"].duplicated(keep=False)]
    .sort_values("url")
)

print(f"Duplicate rows: {len(duplicates)}")
display(duplicates)

Duplicate rows: 0


,unique_id,url,name,license_number,license_expiry_date,broker_type,mobile_number,email_address,the_unified_number_of_the_establishment,broker_licenses,street,city,region,district,building_number,additional_number,scraped_ts,date,iteration_number


In [47]:
date_regex = r"^\d{4}-\d{2}-\d{2}$"

for col in ["scraped_ts", "date"]:
    invalid = df[
        ~df[col].fillna("").astype(str).str.match(date_regex)
    ]
    print(col, len(invalid))

scraped_ts 0
date 0


In [50]:
df.name.value_counts()

name
AL DHIAFA REAL ESTATE SELLING BUYING AND RENTIN - L.L.C                                                                        1
(AL DOR SELLING & BUYING & RENTING REALESTATE (L.L.C                                                                           1
ARDH ALNAKHEEL PROPERTIES L.L.C                                                                                                1
QSA  PROPERTIES L.L.C                                                                                                          1
DREAM STREET PROPERTIES L.L.C                                                                                                  1
DARNA PROPERTIES L.L.C                                                                                                         1
ALBAYT BAYTUK PROPERTIES L.L.C                                                                                                 1
AUDAZ PROPERTIES L.L.C                                                                      

In [38]:
broker = df["broker"].fillna("").astype(str).str.strip()
display_name = df["broker_display_name"].fillna("").astype(str).str.strip()

invalid = df[
    broker != display_name.str.upper()
]

display(invalid[["id", "broker", "broker_display_name"]])

KeyError: 'broker'

In [51]:
df[df.name=="(AL DOR SELLING & BUYING & RENTING REALESTATE (L.L.C"].url

0    https://eservices.ajmanded.ae/en/TradeLicense/License/25b4e718-6633-473f-95be-f1ae2f780b46
Name: url, dtype: object

In [40]:
import pandas as pd

invalid = df[
   
     df["developer_start_price"].replace(r'^\s*$', pd.NA, regex=True).isna() 
]

print(f"Rows where both prices are missing: {len(invalid)}")
display(invalid)

KeyError: 'developer_start_price'

In [55]:
df.license_expiry_date.value_counts()


license_expiry_date
05/01/2027    10
19/05/2027    10
24/02/2027     9
25/02/2027     9
12/01/2027     8
22/02/2027     8
03/08/2027     8
16/02/2027     8
26/04/2027     8
09/12/2026     8
04/02/2027     8
28/01/2027     7
11/02/2027     7
22/04/2027     7
16/12/2026     7
21/12/2026     7
17/03/2027     7
09/02/2027     7
09/01/2027     7
26/07/2027     7
11/08/2027     6
01/07/2027     6
15/12/2026     6
29/07/2027     6
11/05/2027     6
31/03/2027     6
27/07/2027     6
11/01/2027     6
01/06/2027     6
07/01/2027     6
05/02/2027     6
09/11/2026     6
19/01/2027     6
09/07/2027     6
13/01/2027     6
16/06/2027     6
19/08/2027     6
12/02/2027     6
19/11/2026     6
19/10/2027     6
10/11/2026     6
20/05/2027     6
23/04/2027     6
24/06/2027     5
10/03/2027     5
11/11/2026     5
21/01/2027     5
26/08/2027     5
18/01/2027     5
29/01/2027     5
03/02/2027     5
10/02/2027     5
05/08/2027     5
21/02/2027     5
03/01/2027     5
21/04/2027     5
30/04/2027     5
12/03/2027 

In [57]:
import json
import pandas as pd

file_path = "/home/user/Downloads/emlakjet_tur_2026_11.json"


# Stream JSON records without loading the entire file
def stream_json_objects(file_path):
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        buffer = ""

        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            buffer += chunk

            while True:
                buffer = buffer.lstrip()

                if not buffer:
                    break

                try:
                    record, index = decoder.raw_decode(buffer)
                    yield record
                    buffer = buffer[index:]

                except json.JSONDecodeError:
                    break


# Check licences expired more than 2 months before iteration date
issues = []

for record_no, record in enumerate(
    stream_json_objects(file_path), start=1
):
    if not isinstance(record, dict):
        continue

    expiry_date = pd.to_datetime(
        record.get("license_expiry_date"), errors="coerce"
    )

    iteration = str(record.get("iteration_number", "")).strip()

    try:
        iteration_date = pd.to_datetime(
            iteration + "_01",
            format="%Y_%m_%d",
            errors="raise"
        )
    except (ValueError, TypeError):
        continue

    if pd.isna(expiry_date):
        continue

    if expiry_date < iteration_date - pd.DateOffset(months=2):
        issues.append({
            "record_no": record_no,
            "unique_id": record.get("unique_id"),
            "url": record.get("url"),
            "license_expiry_date": expiry_date.date(),
            "iteration_date": iteration_date.date()
        })

    if record_no % 100000 == 0:
        print(f"Processed {record_no:,} records")


# Display results
result = pd.DataFrame(issues)

print(f"\nExpired licences found: {len(result):,}")
display(result)

FileNotFoundError: [Errno 2] No such file or directory: '/home/user/Downloads/emlakjet_tur_2026_11.json'

In [30]:
# 2. Define the columns
columns_to_analyze = [
    "category", "title", "location", "price", "currency", "price_per",
    "bedrooms", "bathrooms", "furnished", "rera_permit_number",
    "dtcm_licence", "scraped_ts", "amenities", "details", "agent_name",
    "number_of_photos", "user_id", "phone_number", "date", "iteration_number",
    "depth", "sub_category_1", "property_type", "sub_category_2",
    "published_at", "listing_availability", "ownership", "verified_business"
]

# 3. Print the number of unique entries and their breakdown
for col in columns_to_analyze:
    if col in df.columns:
        unique_count = df[col].nunique()
        print(f"==================================================")
        print(f"COLUMN: {col}")
        print(f"Total Unique Values: {unique_count}")
        print(f"==================================================")
        print(df[col].value_counts())
        print("\n")

COLUMN: category
Total Unique Values: 2
category
sale    3990
rent     627
Name: count, dtype: int64


COLUMN: title
Total Unique Values: 28
title
أرض سكنية للبيع                          2543
فيلا سكنية للبيع                          727
شقة سكنية للإيجار                         447
شقة سكنية للبيع                           288
أرض تجارية للبيع                          236
مبنى تجاري للبيع                           77
فيلا سكنية للإيجار                         59
شقة استوديو سكنية للإيجار                  55
شقة استوديو سكنية للبيع                    20
Residential Villa For Rent                 16
Residential Land For Sale                  16
Residential Villa For Sale                 16
Commercial Office For Rent                 16
Commercial Land For Sale                   16
Commercial Building For Sale               16
Residential Apartment For Sale             14
Residential Apartment For Rent             14
مكتب تجاري للإيجار                          8
Residential Penthouse For

In [58]:
import json
import pandas as pd

file_path = "/home/user/Downloads/ajmanded_uae_2026_11.json"


# Stream JSON records without loading the entire file
def stream_json_objects(file_path):
    decoder = json.JSONDecoder()

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        buffer = ""

        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            buffer += chunk

            while True:
                buffer = buffer.lstrip()

                if not buffer:
                    break

                try:
                    record, index = decoder.raw_decode(buffer)
                    yield record
                    buffer = buffer[index:]

                except json.JSONDecodeError:
                    break


# --------------------------------------------------
# Check licence expiry
# --------------------------------------------------
issues = []
record_count = 0

for record_no, record in enumerate(
    stream_json_objects(file_path), start=1
):
    record_count += 1

    if not isinstance(record, dict):
        continue

    expiry_date = pd.to_datetime(
        record.get("license_expiry_date"),
        errors="coerce"
    )

    iteration = str(
        record.get("iteration_number", "")
    ).strip()

    # Convert YYYY_MM into a date
    try:
        iteration_date = pd.to_datetime(
            iteration + "_01",
            format="%Y_%m_%d",
            errors="raise"
        )
    except (ValueError, TypeError):
        continue

    if pd.isna(expiry_date):
        continue

    # Licence expired more than 2 months before iteration date
    cutoff_date = iteration_date - pd.DateOffset(months=2)

    if expiry_date < cutoff_date:
        issues.append({
            "record_no": record_no,
            "unique_id": record.get("unique_id"),
            "url": record.get("url"),
            "license_expiry_date": expiry_date.date(),
            "iteration_date": iteration_date.date()
        })

    if record_count % 100000 == 0:
        print(f"Processed {record_count:,} records")


# --------------------------------------------------
# Display results
# --------------------------------------------------
result = pd.DataFrame(issues)

print("\n========== QA RESULT ==========")
print(f"Total records checked: {record_count:,}")
print(f"Expired licences found: {len(result):,}")

if result.empty:
    print("✓ No issues found")
else:
    print("✗ Expired licences found")
    display(result)

/tmp/ipykernel_14475/1666967976.py:51: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  expiry_date = pd.to_datetime(



========== QA RESULT ==========
Total records checked: 1,138
Expired licences found: 45
✗ Expired licences found


,record_no,unique_id,url,license_expiry_date,iteration_date
0,71,7e0ff999-b11a-49cd-8799-08ce1d853304,https://eservices.ajmanded.ae/en/TradeLicense/License/7e0ff999-b11a-49cd-8799-08ce1d853304,2026-08-11,2026-11-01
1,73,6fa67d3b-67d5-4eef-a141-6e237405c3cd,https://eservices.ajmanded.ae/en/TradeLicense/License/6fa67d3b-67d5-4eef-a141-6e237405c3cd,2026-01-09,2026-11-01
2,84,ae7ac71b-88d9-4ad8-b664-6a7cd2247c00,https://eservices.ajmanded.ae/en/TradeLicense/License/ae7ac71b-88d9-4ad8-b664-6a7cd2247c00,2026-02-12,2026-11-01
3,87,ea75908a-fc5e-4790-9650-44069e898bd9,https://eservices.ajmanded.ae/en/TradeLicense/License/ea75908a-fc5e-4790-9650-44069e898bd9,2026-02-12,2026-11-01
4,90,ab1e41dd-2981-4805-944d-69faa5545bee,https://eservices.ajmanded.ae/en/TradeLicense/License/ab1e41dd-2981-4805-944d-69faa5545bee,2026-04-12,2026-11-01
5,108,e30da2b3-384f-4be2-adb1-8c6b12437144,https://eservices.ajmanded.ae/en/TradeLicense/License/e30da2b3-384f-4be2-adb1-8c6b12437144,2026-03-10,2026-11-01
6,114,bd566e6e-95d2-45de-9df7-82cde59e6b85,https://eservices.ajmanded.ae/en/TradeLicense/License/bd566e6e-95d2-45de-9df7-82cde59e6b85,2026-02-11,2026-11-01
7,124,f2437b22-ce7f-493a-97b8-0920462b0211,https://eservices.ajmanded.ae/en/TradeLicense/License/f2437b22-ce7f-493a-97b8-0920462b0211,2026-02-10,2026-11-01
8,186,2e9325ee-2925-43d4-9308-926b408bcf0f,https://eservices.ajmanded.ae/en/TradeLicense/License/2e9325ee-2925-43d4-9308-926b408bcf0f,2026-07-11,2026-11-01
9,209,4c3cb81e-8324-4b96-bb18-4232b9cbd274,https://eservices.ajmanded.ae/en/TradeLicense/License/4c3cb81e-8324-4b96-bb18-4232b9cbd274,2026-04-11,2026-11-01


In [27]:
df.property_type.value_counts()

property_type
أرض          2806
فيلا          778
شقة           737
مبنى           79
استوديو        76
Villa          41
Land           34
Apartment      28
Office         20
Building       19
Penthouse      11
مكتب            8
Studio          4
Name: count, dtype: int64

In [24]:
df.nunique()

reference_number       3880
url                    4641
broker_display_name      29
broker                   29
category                  2
category_url              2
title                    28
description            4134
location                176
price                  2230
currency                  1
price_per                 1
bedrooms                  8
bathrooms                16
furnished                 7
rera_permit_number        1
dtcm_licence              1
scraped_ts                1
amenities              1463
details                1917
agent_name               94
number_of_photos          3
phone_number             54
user_id                   1
date                      1
iteration_number          1
sub_category_1            4
sub_category_2            1
property_type            13
depth                     2
last_update             138
published_at            662
id                     4641
dtype: int64

In [59]:
import pandas as pd

date_cols = [
    "scraped_ts",
    "license_start_date",
    "license_end_date",
    "date"
]

for col in date_cols:
    # Convert to datetime using the expected format
    parsed = pd.to_datetime(df[col], format="%Y-%m-%d", errors="coerce")

    invalid = df[parsed.isna() & df[col].notna() & (df[col].astype(str).str.strip() != "")]

    print(f"\n{col}")
    print(f"Invalid date format count: {len(invalid)}")

    if len(invalid) > 0:
        display(invalid[[col]])


scraped_ts
Invalid date format count: 0


KeyError: 'license_start_date'

In [60]:
df.isnull().sum()

reference_number       0
id                     0
url                    0
broker_display_name    0
broker                 0
category               0
category_url           0
title                  0
description            0
location               0
price                  0
currency               0
price_per              0
bedrooms               0
bathrooms              0
furnished              0
rera_permit_number     0
dtcm_licence           0
scraped_ts             0
amenities              0
details                0
agent_name             0
number_of_photos       0
user_id                0
phone_number           0
date                   0
iteration_number       0
depth                  0
sub_category_1         0
property_type          0
sub_category_2         0
published_at           0
dtype: int64

In [61]:
null_counts = df.isna().sum()

print(null_counts)

reference_number       0
id                     0
url                    0
broker_display_name    0
broker                 0
category               0
category_url           0
title                  0
description            0
location               0
price                  0
currency               0
price_per              0
bedrooms               0
bathrooms              0
furnished              0
rera_permit_number     0
dtcm_licence           0
scraped_ts             0
amenities              0
details                0
agent_name             0
number_of_photos       0
user_id                0
phone_number           0
date                   0
iteration_number       0
depth                  0
sub_category_1         0
property_type          0
sub_category_2         0
published_at           0
dtype: int64


In [17]:
empty_counts = (
    df.astype(str)
      .apply(lambda col: (col.str.strip() == "").sum())
)

print(empty_counts)

profile_url                 0
first_name                  0
middle_name             36159
last_name                 198
image_url                9884
office_name                 0
address                     0
description             21987
languages                   0
social                      0
website                 26796
email                       0
title                    5764
country                     0
city                        0
zipcode                     0
state                       0
agent_phone_numbers         0
office_phone_numbers        0
dtype: int64


In [63]:
df[df["phone_number_1"].astype(str).str.strip() == ""]

KeyError: 'phone_number_1'

In [76]:
import json

file_path = r"/home/user/Downloads/dari_ae_2025_07_04.json"

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            json.loads(line)
        except Exception as e:
            print(f"Error at line {i}")
            print(e)
            print(line[:500])   # show the beginning of the problematic line
            break
    else:
        print("✅ No JSON syntax errors found.")

FileNotFoundError: [Errno 2] No such file or directory: '/home/user/Downloads/dari_ae_2025_07_04.json'

In [65]:
import json

file_path = r"/home/user/Downloads/dari_ae_2025_07_04.json"

with open(file_path, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)
        print("✅ Valid JSON")
    except Exception as e:
        print(e)

Extra data: line 2 column 1 (char 498)


In [39]:
cols = ["phone_number_1", "phone_number_2"]

df[cols] = df[cols].fillna("").astype(str).apply(lambda x: x.str.strip())

same_phone = df[
    (df["phone_number_1"] != "") &
    (df["phone_number_1"] == df["phone_number_2"])
]

print("Records with same phone_number_1 and phone_number_2:", len(same_phone))

display(same_phone[["url", "name", "phone_number_1", "phone_number_2"]])

KeyError: "None of [Index(['phone_number_1', 'phone_number_2'], dtype='object')] are in the [columns]"

In [22]:
df.scraped_ts.value_counts()

scraped_ts
2026-10-27    34967
Name: count, dtype: int64

In [23]:
df.date.value_counts()

date
2026-10-27    34967
Name: count, dtype: int64

In [24]:
df.iteration_number.value_counts()

iteration_number
202611    34967
Name: count, dtype: int64

In [25]:
df.category.value_counts()

category
sale      20424
rent      13886
waiver      657
Name: count, dtype: int64

In [42]:
df[df.price == ""].url

2683                                https://www.nawy.com/compound/1261-hacienda-waters/property/137713-chalet-for-sale-in-hacienda-waters-with-1-bedrooms-in-ras-el-hekma-by-palm-hills-developments
7478                                                https://www.nawy.com/compound/745-eastshire/property/29641-townhouse-for-sale-in-eastshire-with-7-bedrooms-in-new-cairo-by-al-qamzi-developments
12830    https://www.nawy.com/compound/1904-spire-club-hills/property/132560-apartment-for-sale-in-spire-club-hills-with-21-bedrooms-in-northern-expansion-by-housing-and-development-properties-hdp
14595                                          https://www.nawy.com/compound/664-the-estates-residence/property/56519-duplex-for-sale-in-the-estates-residence-with-3-bedrooms-in-new-zayed-by-sodic
14891                                     https://www.nawy.com/compound/664-the-estates-residence/property/115256-apartment-for-sale-in-the-estates-residence-with-21-bedrooms-in-new-zayed-by-sodic
17746          

In [26]:
df.bathrooms.value_counts()

bathrooms
    34967
Name: count, dtype: int64

In [18]:
import pandas as pd

for col in df.columns:
    print("\n" + "="*80)
    print(f"Column: {col}")
    print("="*80)

    value_counts = (
        df[col]
        .fillna("<NULL>")        # Show nulls explicitly
        .astype(str)
        .value_counts(dropna=False)
    )

    print(value_counts)


Column: profile_url
profile_url
https://www.century21.com/sc/spartanburg/agents/sara-suarez/aid-P00200000GBE1ydM9DaKYIj9O1T0ontQDtoSsRuu    1
https://www.century21.com/al/athens/agents/jan-brown/aid-P00200000FDdrFjv5h1wCldBmeEsY1uBX0Aegkum           1
https://www.century21.com/al/athens/agents/tammy-pressnell/aid-P00200000FDdrS9YdAW9WXtTwXsvuIVuAruP1HTr     1
https://www.century21.com/al/athens/agents/roger-mcelyea/aid-P00200000GeBNb8dCtERX2EOd77UOl5q8EBKCLqM       1
https://www.century21.com/al/athens/agents/luther-smith/aid-P00200000FDdsKwZymjn4OZSr3haOAbJj18v28Q1        1
                                                                                                           ..
https://www.century21.com/al/huntsville/agents/misty-guzman/aid-P00200000GZKzQg4UW5bqCf60GTCyX7a4Il0xbEP    1
https://www.century21.com/al/huntsville/agents/jesus-garcia/aid-P00200000GpQBlEw38uRNRV0pz4ubm4wnU2OmmHg    1
https://www.century21.com/al/huntsville/agents/shawn-guzman/aid-P00200000GZKpppNuuTL1eF

In [42]:
df.scraped_ts.value_counts()

scraped_ts
2026-10-26    1077
Name: count, dtype: int64

In [43]:
df.date.value_counts()

date
2026-10-26    1077
Name: count, dtype: int64

In [44]:
df.published_at.value_counts()

AttributeError: 'DataFrame' object has no attribute 'published_at'

In [45]:
df.depth.value_counts()

AttributeError: 'DataFrame' object has no attribute 'depth'

In [46]:
df.dtypes

url                                    object
name                                   object
profession                             object
trade_license_number                   object
license_start_date                     object
license_end_date                       object
registration_license_number            object
email                                  object
phone_number_1                         object
phone_number_2                         object
phone_number_3                         object
iteration_number                        int64
scraped_ts                             object
date                           datetime64[ns]
company_employees                      object
dtype: object

In [52]:
import re
import pandas as pd

issues = []

for col in df.columns:

    s = df[col].fillna("").astype(str)

    # Check leading, trailing, multiple spaces, tab, newline
    mask = (
        s.str.contains(r"^\s", regex=True) |
        s.str.contains(r"\s$", regex=True) |
        s.str.contains(r" {2,}", regex=True) |
        s.str.contains(r"\t", regex=True) |
        s.str.contains(r"\n", regex=True) |
        s.str.contains(r"\r", regex=True)
    )

    # Only the first issue from each field
    if mask.any():
        idx = df.index[mask][0]

        issues.append({
            "record_no": idx + 1,
            "url": df.loc[idx, "url"],
            "issue_field": col
        })

result = pd.DataFrame(issues)

print("Fields with whitespace issues:", len(result))
print(result.to_string(index=False))

Fields with whitespace issues: 2
 record_no                                                                                        url     issue_field
        75 https://eservices.ajmanded.ae/en/TradeLicense/License/65685802-2b02-450f-a049-ffb939a68d30            name
        29 https://eservices.ajmanded.ae/en/TradeLicense/License/fd16341b-0fd3-4174-b9af-7d6caa78fec8 building_number


In [ ]:
df.web.value_counts()

number_of_photos
3      56443
1      33744
4      32441
5      26902
6      22942
       ...  
181        1
76         1
126        1
110        1
138        1
Name: count, Length: 127, dtype: int64

In [77]:
df[df.bathrooms== "2.5"].url

Series([], Name: url, dtype: object)

In [23]:
df[df.number_of_photos == ""].url

59     https://bh.opensooq.com/en/search/288233222
119    https://bh.opensooq.com/en/search/286653756
226    https://bh.opensooq.com/en/search/288218214
252    https://bh.opensooq.com/en/search/287390796
600    https://bh.opensooq.com/en/search/288081992
640    https://bh.opensooq.com/en/search/288629646
641    https://bh.opensooq.com/en/search/289086966
721    https://bh.opensooq.com/en/search/288614478
757    https://bh.opensooq.com/en/search/288502712
Name: url, dtype: object

In [48]:
df[df.country =="Jamaica"].profile_url

7399    https://www.bhgre.com/jam/city-unavailable/agents/better-homes-and-gardens-real-estate-jamaica/aid-P00200000FDdscdXBZtMRb3oSgOQD9P9S6pLBP1I
7400                                      https://www.bhgre.com/jam/city-unavailable/agents/edwin-wint/aid-P00200000FDdqSekqCfO4OyzqGu2mbNGsFTEcVYL
Name: profile_url, dtype: object